In [ ]:
import ee
import pandas as pd
from datetime import datetime

# 初始化 GEE
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# 配置参数
TIME_WINDOW_HOURS = 6 
MIN_OVERLAP_RATIO = 0.5  # 现在指：光学影像有 50% 以上落在 S1 范围内

from datetime import datetime, timezone

def get_matches_for_s1(s1_id, time_window_hours):
    """
    1. 修正 Date.difference 单位报错
    2. 严格使用秒级 (second) 校验时间窗口
    3. 保存完整 Asset ID 并实现传感器去重
    """
    try:
        # 1. 加载 S1 影像并获取其基础信息
        # 自动补全路径前缀以增强健壮性
        s1_path = s1_id if '/' in s1_id else f'COPERNICUS/S1_GRD/{s1_id}'
        s1_img = ee.Image(s1_path)
        s1_geom = s1_img.geometry()
        s1_time = s1_img.date()
        
        # 定义最大允许的时间差（秒）
        max_diff_seconds = time_window_hours * 3600
        
        # 2. 定义光学影像集合路径
        collections = {
            'Landsat9': "LANDSAT/LC09/C02/T1_L2",
            'Landsat8': "LANDSAT/LC08/C02/T1_L2",
            'Landsat7': "LANDSAT/LE07/C02/T1_L2",
            'Sentinel2': "COPERNICUS/S2_HARMONIZED"
        }

        all_sensor_matches = []
        
        # 3. 逐个集合处理，确保 ID 不被 merge 污染
        for sensor_name, asset_path in collections.items():
            col = ee.ImageCollection(asset_path) \
                .filterBounds(s1_geom) \
                .filterDate(s1_time.advance(-time_window_hours - 1, 'hour'), 
                            s1_time.advance(time_window_hours + 1, 'hour'))
            
            def process_opt(img):
                img = ee.Image(img)
                opt_time = img.date()
                
                # --- 修正点：将 'milli' 改为 'second' ---
                time_diff_sec = ee.Number(opt_time.difference(s1_time, 'second')).abs()
                
                # 计算重叠率：光学影像落在 S1 内部的比例
                opt_geom = img.geometry()
                intersection = opt_geom.intersection(s1_geom, 100)
                overlap = intersection.area(100).divide(opt_geom.area(100))
                
                # 拼接可以直接用于 ee.Image() 的 Asset ID
                full_asset_id = ee.String(asset_path).cat('/').cat(img.get('system:index'))
                
                return img.set({
                    'overlap_ratio': overlap,
                    'time_diff_sec': time_diff_sec,
                    'full_asset_id': full_asset_id,
                    'sensor_label': sensor_name,
                    's1_time_str': s1_time.format('yyyy-MM-dd HH:mm:ss')
                })

            # 应用严格过滤：秒级时间差 + 重叠率阈值
            matched = col.map(process_opt) \
                .filter(ee.Filter.lte('time_diff_sec', max_diff_seconds)) \
                .filter(ee.Filter.gte('overlap_ratio', MIN_OVERLAP_RATIO)) \
                .sort('overlap_ratio', False) # 保留重叠面积最大的
            
            # 每个传感器只取 top 1 匹配项
            top_match = matched.limit(1).getInfo()
            
            if top_match['features']:
                props = top_match['features'][0]['properties']
                # 获取 UTC 时间戳并转换
                raw_ms = top_match['features'][0]['properties']['system:time_start']
                # 明确指定 tzinfo=timezone.utc 避免本地时区干扰
                m_time = datetime.fromtimestamp(raw_ms/1000.0, tz=timezone.utc)
                
                # 本地计算时间差作为记录
                s_time = datetime.strptime(props['s1_time_str'], '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc)
                diff_hours = round(abs((m_time - s_time).total_seconds() / 3600), 2)

                all_sensor_matches.append({
                    's1_scene': s1_id,
                    'matched_id': props['full_asset_id'], # 格式：LANDSAT/LC08/C02/T1_L2/...
                    'matched_sensor': props['sensor_label'],
                    'overlap_ratio': round(props['overlap_ratio'], 3),
                    'time_diff_hours': diff_hours,
                    's1_time': props['s1_time_str'],
                    'matched_time': m_time.strftime('%Y-%m-%d %H:%M:%S')
                })

        return all_sensor_matches

    except Exception as e:
        print(f"  [Error] {s1_id}: {e}")
        return []
    
if __name__ == "__main__":
    # --- 文件路径配置 ---
    input_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\overlap_S1name.csv"
    output_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\overlapping_S1_MultiSensor_all.csv"
    
    df = pd.read_csv(input_csv)
    all_results = []

    print(f"开始处理，模式：光学影像覆盖率 > {MIN_OVERLAP_RATIO*100}%")

    for idx, row in df.iterrows():
        s1_scene = row['scene_name']
        matches = get_matches_for_s1(s1_scene, TIME_WINDOW_HOURS)
        
        if matches:
            for m in matches:
                # 完善本地结果
                t1 = datetime.strptime(m['s1_time'], '%Y-%m-%d %H:%M:%S')
                t2 = datetime.strptime(m['matched_time'], '%Y-%m-%d %H:%M:%S')
                m['time_diff_hours'] = round(abs((t1 - t2).total_seconds() / 3600), 2)
                m['s1_scene'] = s1_scene
                all_results.append(m)
            print(f"[{idx+1}/{len(df)}] {s1_scene} -> 找到 {len(matches)} 个匹配项")
        else:
            print(f"[{idx+1}/{len(df)}] {s1_scene} -> 无匹配")

        # 实时保存进度
        if (idx + 1) % 10 == 0 and all_results:
            pd.DataFrame(all_results).to_csv(output_csv, index=False)

    # 最终保存
    if all_results:
        final_df = pd.DataFrame(all_results)
        final_df.to_csv(output_csv, index=False)
        print(f"\n任务完成！共找到 {len(all_results)} 对匹配影像。")

开始处理，模式：光学影像覆盖率 > 50.0%
[1/593] S1A_EW_GRDM_1SDH_20150107T215431_20150107T215531_004071_004EAF_D49C -> 无匹配
[2/593] S1A_EW_GRDM_1SDH_20150624T120841_20150624T120945_006515_008A71_E9F5 -> 找到 1 个匹配项
[3/593] S1A_EW_GRDM_1SDH_20150623T112721_20150623T112821_006500_008A04_992B -> 无匹配
[4/593] S1A_EW_GRDM_1SDH_20150622T122515_20150622T122615_006486_00899E_D31E -> 找到 1 个匹配项
[5/593] S1A_EW_GRDM_1SDH_20150614T115242_20150614T115342_006369_008647_9AA3 -> 找到 1 个匹配项
[6/593] S1A_EW_GRDM_1SDH_20150612T134725_20150612T134830_006341_00857A_131E -> 找到 1 个匹配项
[7/593] S1A_EW_GRDM_1SDH_20150612T120840_20150612T120945_006340_008573_ACFE -> 找到 1 个匹配项
[8/593] S1A_EW_GRDM_1SDH_20150607T120150_20150607T120250_006267_00835B_332E -> 无匹配
[9/593] S1A_EW_GRDM_1SDH_20150605T121804_20150605T121904_006238_00827D_6A5C -> 无匹配
[10/593] S1A_EW_GRDM_1SDH_20150604T131534_20150604T131634_006224_008208_49CD -> 找到 1 个匹配项
[11/593] S1A_EW_GRDM_1SDH_20150527T124229_20150527T124329_006107_007EC8_08A6 -> 无匹配
[12/593] S1A_EW_GRDM_1SDH

In [3]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# 初始化 GEE
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# 配置参数
TIME_WINDOW_HOURS = 6 
MIN_OVERLAP_RATIO = 0.3 

def get_matches_for_s1(s1_id, time_window):
    """
    在服务器端一次性完成几何获取、时间筛选和重叠度计算
    """
    try:
        # 1. 加载 S1 影像 (确保路径正确)
        s1_path = f'COPERNICUS/S1_GRD/{s1_id}' if not s1_id.startswith('COPERNICUS') else s1_id
        s1_img = ee.Image(s1_path)
        s1_geom = s1_img.geometry()
        s1_date = s1_img.date()
        
        # 2. 定义时间范围
        start_date = s1_date.advance(-time_window, 'hour')
        end_date = s1_date.advance(time_window, 'hour')
        
        # 3. 定义重叠计算函数 (Server-side)
        def check_overlap(opt_img):
            opt_img = ee.Image(opt_img)
            intersection = opt_img.geometry().intersection(s1_geom, 10)
            overlap = intersection.area(10).divide(s1_geom.area(10))
            return opt_img.set({
                'overlap_ratio': overlap,
                's1_time_str': s1_date.format('yyyy-MM-dd HH:mm:ss')
            })

        # 4. 获取并过滤光学影像
        # 合并 L8 和 S2 集合
        l8_col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate(start_date, end_date).filterBounds(s1_geom)
        s2_col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterDate(start_date, end_date).filterBounds(s1_geom)
        
        # 统一处理
        merged_col = l8_col.merge(s2_col).map(check_overlap)
        
        # 仅保留满足重叠率条件的影像
        matching_col = merged_col.filter(ee.Filter.gte('overlap_ratio', MIN_OVERLAP_RATIO))
        
        # 5. 提取必要属性传回本地 (仅此一步是网络传输)
        def extract_info(img):
            return ee.Feature(None, {
                'matched_id': img.get('system:index'),
                'matched_sensor': ee.Algorithms.If(ee.String(img.get('system:index')).slice(0,2).equals('LC'), 'Landsat8', 'Sentinel2'),
                'overlap_ratio': img.get('overlap_ratio'),
                'matched_time': ee.Date(img.get('system:time_start')).format('yyyy-MM-dd HH:mm:ss'),
                's1_time': img.get('s1_time_str')
            })

        results = matching_col.map(extract_info).getInfo()
        return [feat['properties'] for feat in results['features']]

    except Exception as e:
        print(f"  [Error] 处理 {s1_id} 时出错: {e}")
        return []



In [4]:


if __name__ == "__main__":
    # 读取数据
    input_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\statistic_20251231\overall_density_summary_2023_raster_filtered_detailed.csv"
    output_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\overlapping_S1_Optical_v2.csv"
    
    df = pd.read_csv(input_csv)
    all_results = []

    print(f"开始处理，共 {len(df)} 条记录...")

    for idx, row in df.iterrows():
        s1_scene = row['scene_name']
        print(f"[{idx+1}/{len(df)}] 正在检索: {s1_scene}")
        
        matches = get_matches_for_s1(s1_scene, TIME_WINDOW_HOURS)
        
        for m in matches:
            # 计算时间差 (小时)
            t1 = datetime.strptime(m['s1_time'], '%Y-%m-%d %H:%M:%S')
            t2 = datetime.strptime(m['matched_time'], '%Y-%m-%d %H:%M:%S')
            m['time_diff_hours'] = round(abs((t1 - t2).total_seconds() / 3600), 2)
            m['s1_scene'] = s1_scene
            all_results.append(m)

        # 每 20 条记录保存一次进度，防止崩溃丢失数据
        if (idx + 1) % 20 == 0:
            pd.DataFrame(all_results).to_csv(output_csv, index=False)
            print(f"--- 已自动保存进度至 {output_csv} ---")

    # 最终保存
    if all_results:
        result_df = pd.DataFrame(all_results)
        # 调整列顺序
        cols = ['s1_scene', 's1_time', 'matched_sensor', 'matched_id', 'matched_time', 'time_diff_hours', 'overlap_ratio']
        result_df[cols].to_csv(output_csv, index=False)
        print(f"\n任务完成！匹配到 {len(all_results)} 对影像。")
    else:
        print("\n未找到任何匹配结果。")

开始处理，共 51 条记录...
[1/51] 正在检索: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
[2/51] 正在检索: S1A_EW_GRDM_1SDH_20230626T142248_20230626T142348_049158_05E944_BA88
[3/51] 正在检索: S1A_EW_GRDM_1SDH_20230626T142148_20230626T142248_049158_05E944_ECF8
[4/51] 正在检索: S1A_EW_GRDM_1SDH_20230624T143817_20230624T143917_049129_05E859_41EE
[5/51] 正在检索: S1A_EW_GRDM_1SDH_20230621T123356_20230621T123500_049084_05E706_4FAD
[6/51] 正在检索: S1A_EW_GRDM_1SDH_20230617T144728_20230617T144811_049027_05E552_F861
[7/51] 正在检索: S1A_EW_GRDM_1SDH_20230616T140522_20230616T140622_049012_05E4D5_75E8
[8/51] 正在检索: S1A_EW_GRDM_1SDH_20230615T114457_20230615T114601_048996_05E45F_7371
[9/51] 正在检索: S1A_EW_GRDM_1SDH_20230614T142148_20230614T142248_048983_05E3F6_1872
[10/51] 正在检索: S1A_EW_GRDM_1SDH_20230613T120119_20230613T120224_048967_05E379_A4FA
[11/51] 正在检索: S1A_EW_GRDM_1SDH_20230611T121733_20230611T121837_048938_05E287_B229
[12/51] 正在检索: S1A_EW_GRDM_1SDH_20230606T134852_20230606T134952_048866_05E060_3E2D
[13/51] 